# <mark>UBC Ovarian Cancer Subtype Classification and Outlier Detection (UBC-OCEAN) - EDA</mark>
<span style="font-size:22px;color:purple"> Thank you for having a look at my notebook - advice and feedback always welcomed!</span>


<div class="alert alert-block alert-info" style="font-size:14px; font-family:verdana;">
    📌 Dataset Link: <a href="https://www.kaggle.com/competitions/UBC-OCEAN/data">https://www.kaggle.com/competitions/UBC-OCEAN/data</a>
</div>


## **Exploratory Data Analysis (EDA)** 
EDA is a crucial step in understanding and preparing your data for any data analysis or machine learning project, including UBC Ovarian Cancer Subtype Classification and Outlier Detection (UBC-OCEAN). Here's a step-by-step guide on how to perform an EDA for this dataset:

**Overview**

The goal of the UBC Ovarian Cancer subtypE clAssification and outlier detectioN (UBC-OCEAN) competition is to classify ovarian cancer subtypes. You will build a model trained on the world's most extensive ovarian cancer dataset of histopathology images obtained from more than 20 medical centers.


**Data Collection:**

Begin by obtaining the UBC-OCEAN dataset, which should include information on ovarian cancer subtypes and possibly outlier detection data. Ensure you have a clear understanding of the dataset's structure and the meaning of each variable.
Data Loading:

Import the dataset into your preferred data analysis environment, such as Python with libraries like pandas, numpy, and matplotlib/seaborn for visualization.

**Data Loading:**

Import the dataset into your preferred data analysis environment, such as Python with libraries like pandas, numpy, and matplotlib/seaborn for visualization.





### **Initial Exploration:**

**1 - Start by examining the basic characteristics of the data:**

    Check the first few rows using df.head().
    Check the data types and missing values using df.info().
    Calculate basic statistics using df.describe().
    Data Cleaning:

**2 - Handle missing values, outliers, and duplicates:**

    Use techniques like imputation for missing values.
    Identify and deal with outliers appropriately.
    Remove duplicate rows if necessary.
    
**3 - Data Visualization:**

    Create visualizations to gain insights into the data:
    Histograms and box plots for numerical features.
    Bar plots for categorical features.
    Correlation matrix and scatter plots to understand relationships between variables.
    
**4 - Feature Analysis:**

    Explore relationships between features and the target variable(s) for classification and outlier detection.
    Visualize how different features vary across different subtypes or classes.
    Use box plots, violin plots, or swarm plots to compare feature distributions.

**5 - Outlier Detection:**

    If your dataset contains information related to outlier detection, perform a dedicated EDA for this aspect:
    Visualize outliers using scatter plots or box plots.
    Apply statistical methods or machine learning techniques to identify outliers.

**6 - Dimensionality Reduction (optional):**

    If the dataset has many features, consider dimensionality reduction techniques like Principal Component Analysis (PCA) to reduce the number of variables while preserving important information.

**8 - Summary and Insights:**

    Summarize your findings from the EDA, including any patterns, trends, or anomalies observed.
    Document any data preprocessing steps applied.

**7 - Next Steps:**

    Based on your EDA findings, plan your next steps, which may include feature engineering, model selection, and further data preprocessing.


Remember that EDA is an iterative process, and you may need to revisit these steps as you delve deeper into the dataset and develop your machine learning or data analysis models.

In [ ]:
%%capture 
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from skimage import io
import os
import seaborn as sns
import cv2
import random
import os
import glob
import imageio
import gc
import math
import copy
import time

# For data manipulation
import numpy as np
import pandas as pd

# Pytorch Imports
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.optim import lr_scheduler
from torch.utils.data import Dataset, DataLoader
from torch.cuda import amp
import torchvision

# Utils
import joblib
from tqdm import tqdm
from collections import defaultdict

# Sklearn Imports
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold

# For Image Models
import timm

# Albumentations for augmentations
import albumentations as A
from albumentations.pytorch import ToTensorV2

# For colored terminal text
from colorama import Fore, Back, Style
b_ = Fore.BLUE
sr_ = Style.RESET_ALL

import warnings
warnings.filterwarnings("ignore")

# For descriptive error messages
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"

In [ ]:
# Load train data
traindf = pd.read_csv('/kaggle/input/UBC-OCEAN/train.csv')
traindf.head()

In [ ]:
testdf=pd.read_csv('/kaggle/input/UBC-OCEAN/test.csv',dtype=str)
testdf

In [ ]:
traindf['label'].value_counts()

In [ ]:
print(traindf.shape)

In [ ]:
traindf.isna().sum().sum()

## <span style="color:purple">No null values present</span>
​
<div class="alert alert-block alert-info">
<b>Good:</b> now to move onto the next steps </div>

In [ ]:
# list of columns we want the distributions for
columns = [col for col in traindf.columns if col!='label']

# loop to iterate over each column
for col in columns:
    
    # subplot for 3 columns (3 plots)
    fig, axs = plt.subplots(figsize=(15,5), ncols=3)
    
    # 1st plot - distribution of the sample dataset
    sns.histplot(data=traindf, x=col, kde=True, ax=axs[0])
    axs[0].set_title('Sample Distribution')
    
    # 2nd plot - distribution of the selected column where the outcome is 1 (has diabetes)
    sns.histplot(data=traindf[traindf['label']=="HGSC"], x=col, kde=True, ax=axs[1], color='orange')
    axs[1].set_title('label - HGSC')
    
    # 3rd plot - distribution of the selected column where the outcome is 0 (doesn't have diabetes)
    sns.histplot(data=traindf[traindf['label']=="LGSC"], x=col, kde=True, ax=axs[2], color='green')
    axs[2].set_title('label - LGSC')
    
    # showing the plots
    plt.tight_layout()
    plt.show()

In [ ]:
traindf.info()

In [ ]:
print(traindf.image_id.is_unique)
print(traindf.label.is_unique)

Exploring image dimensions refers to understanding and working with the various aspects of an image's size and resolution. In the context of digital images, there are several key dimensions and attributes to consider:

1. **Resolution**: Resolution refers to the number of pixels (individual points of color) contained in an image. It is usually expressed in terms of pixels per inch (PPI) or dots per inch (DPI). Higher resolution images have more detail and are suitable for printing, while lower resolution images may be used for web display or on-screen viewing.

2. **Pixel Dimensions**: Pixel dimensions specify the width and height of an image in pixels. For example, an image might have dimensions of 1920x1080 pixels, which means it is 1920 pixels wide and 1080 pixels tall. This is commonly used for specifying the size of digital images.

3. **Aspect Ratio**: The aspect ratio is the ratio of an image's width to its height. Common aspect ratios include 4:3 (standard television), 16:9 (widescreen television), and 1:1 (square). Maintaining the correct aspect ratio is important to prevent image distortion.

4. **Physical Dimensions**: Physical dimensions refer to the size of an image when printed or displayed in the physical world. This is determined by both the pixel dimensions and the resolution. For example, an image with dimensions of 3000x2000 pixels at 300 DPI will be 10x6.67 inches when printed.

5. **File Size**: The file size of an image is measured in bytes or kilobytes (KB), megabytes (MB), etc. It depends on factors such as the color depth, compression, and pixel dimensions. Larger images with more detail tend to have larger file sizes.

6. **Color Depth**: Color depth, also known as bit depth, determines the number of colors a pixel can represent. Common color depths include 8-bit (256 colors), 24-bit (true color), and 32-bit (true color with alpha channel for transparency).

7. **DPI vs. PPI**: DPI (dots per inch) is often used in the context of printing, indicating how many ink dots a printer can produce in a linear inch. PPI (pixels per inch) is used for screen displays, representing the number of pixels in an inch of screen space.

8. **Scaling**: Scaling an image involves resizing it to different dimensions. You can scale an image up (enlargement) or down (reduction). Be aware that scaling too much can lead to a loss of image quality, especially when making an image larger.

9. **Cropping**: Cropping involves cutting out a portion of an image to focus on a specific area. This changes the pixel dimensions and aspect ratio of the image.

10. **Compression**: Image compression reduces file size by removing redundant or less important data. It can be lossless (no quality loss) or lossy (some quality loss). Common image formats like JPEG use lossy compression.

Understanding and managing these image dimensions and attributes is essential for various purposes, including graphic design, photography, web development, and printing. Depending on your specific needs, you may need to adjust these dimensions and attributes accordingly to achieve the desired result.

In [ ]:
traindf[['image_height', 'image_width']].describe()

In [ ]:
print(traindf.image_id.shape[0])
print(len(os.listdir('/kaggle/input/UBC-OCEAN/train_images')))
print(len(os.listdir('/kaggle/input/UBC-OCEAN/train_thumbnails')))

In [ ]:
#import warnings
#warnings.filterwarnings('ignore')

In [ ]:
traindf.label.value_counts()

In [ ]:
HGSC = traindf[traindf['label']=="HGSC"]
EC = traindf[traindf['label']=="EC"]
CC = traindf[traindf['label']=="CC"]
LGSC = traindf[traindf['label']=="LGSC"]
MC = traindf[traindf['label']=="MC"]

In [ ]:
# Set the figure size
plt.figure(figsize=(20, 6))

# Set the font size
plt.rcParams['font.size'] = 14

# Set the colors
colors = ['lightgreen', 'lightblue', 'purple', 'blue', 'yellow']

# Plot the pie chart for the training set
plt.subplot(1, 1, 1)
plt.pie([len(HGSC), len(EC), len(CC), len(LGSC), len(MC)], labels=['HGSC', 'EC', 'CC', 'LGSC', 'MC'], autopct='%1.1f%%', colors=colors)
plt.title('Training Set')


# Add a main title to the figure
plt.suptitle('Distribution of HGSC, EC, CC, LGSC and MC Images in the Training data', fontsize=20, y=1.05)

# Show the plot
plt.show()

In [ ]:
# Examine the Data Split of training and testing data
train_data = glob.glob('/kaggle/input/UBC-OCEAN/train_images/*.png')
test_data = glob.glob('/kaggle/input/UBC-OCEAN/test_images/*.png')

print(f"The Training Set contains: {len(train_data)} images")
print(f"The Testing Set contains: {len(test_data)} images")

In [ ]:
# Calculate total counts
total_train = len(train_data)
total_test = len(test_data)

# Set the figure size
plt.figure(figsize=(4, 4))

# Set the font size
plt.rcParams['font.size'] = 12

# Set the colors
colors = ['lightgreen', 'red']

# Plot the pie chart for the total set
plt.pie([total_train, total_test], labels=['Training Set', 'Testing Set'], autopct='%1.1f%%', colors=colors)
plt.title('Distribution of Images in Training and Testing Sets')

# Show the plot
plt.show()

## Data Visualization 📈 

### Randomly Visualize Images

In [ ]:
# sample training image

io.imshow('/kaggle/input/UBC-OCEAN/train_thumbnails/10642_thumbnail.png')

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Activation,Conv2D, Flatten, Dropout, MaxPooling2D, BatchNormalization
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from keras import regularizers, optimizers
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.utils.class_weight import compute_class_weight

In [ ]:
traindf=pd.read_csv('/kaggle/input/UBC-OCEAN/train.csv',dtype=str)
traindf

In [ ]:
def append_ext(fn):
    return fn+".png"

def append_ext_thum(fn):
    return fn+"_thumbnail.png"


traindf["image_id_path"]=traindf["image_id"].apply(append_ext)
traindf["image_id_path_thum"]=traindf["image_id"].apply(append_ext_thum)


testdf["image_id_path"]=testdf["image_id"].apply(append_ext)
testdf["image_id_path_thum"]=testdf["image_id"].apply(append_ext_thum)


In [ ]:
testdf['Image_path'] = [os.path.join('/kaggle/input/UBC-OCEAN/test_images', image) for image in testdf['image_id_path']]
testdf['Image_path_thumbnails'] = [os.path.join('/kaggle/input/UBC-OCEAN/test_thumbnails', image) for image in testdf['image_id_path_thum']]
testdf

In [ ]:
traindf['Image_path'] = [os.path.join('/kaggle/input/UBC-OCEAN/train_images', image) for image in traindf['image_id_path']]
traindf['Image_path_thumbnails'] = [os.path.join('/kaggle/input/UBC-OCEAN/train_thumbnails', image) for image in traindf['image_id_path_thum']]
traindf

In [ ]:
full_path_random = np.random.choice(traindf['Image_path_thumbnails'],5)
full_path_random

In [ ]:
from PIL import Image

In [ ]:
def image_viewer(dataset, index, ax):
    image_path =  dataset['Image_path_thumbnails'][index]
    image      =  Image.open(image_path)
    ax.imshow(image)
    
def plot_some_images(dataset, title):
    fig, axs = plt.subplots(nrows = 1,ncols = 2,figsize=(20,8))
    for ind, ax in enumerate(axs.flat):
            index = random.randrange(len(dataset))
            image_viewer(dataset, index, ax)
            ax.set_title(dataset['label'][index], fontsize = 8)
            ax.axis('off')
            fig.suptitle(title, fontsize = 15)
    plt.show()


In [ ]:
#plot_some_images(traindf, 'Trainig Images')

In [ ]:
class_weights = compute_class_weight(class_weight = "balanced",
                                     classes= np.unique(traindf['label']),
                                     y= traindf['label'])

classes = (np.unique(traindf['label']))
class_weights_forplot = dict(zip(classes, class_weights))

In [ ]:
classes

In [ ]:
class_weights_forplot

In [ ]:
class_weights = dict(zip(range(43), class_weights))
class_weights

## **Model Building**


In [ ]:
CONFIG = {
    "seed": 42,
    "img_size": 512,
    "model_name": "tf_efficientnet_b0_ns",
    "num_classes": 5,
    "valid_batch_size": 64,
    "device": torch.device("cuda:0" if torch.cuda.is_available() else "cpu"),
}

In [ ]:
def set_seed(seed=42):
    '''Sets the seed of the entire notebook so results are the same every time we run.
    This is for REPRODUCIBILITY.'''
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    # When running on the CuDNN backend, two further options must be set
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    # Set a fixed value for the hash seed
    os.environ['PYTHONHASHSEED'] = str(seed)
    
set_seed(CONFIG['seed'])

In [ ]:
ROOT_DIR = '/kaggle/input/UBC-OCEAN'
TEST_DIR = '/kaggle/input/UBC-OCEAN/test_thumbnails'
ALT_TEST_DIR = '/kaggle/input/UBC-OCEAN/test_images'
LABEL_ENCODER = "/kaggle/input/ubcpytorchwith-classweights-training-fold1of5/label_encoder.pkl"
BEST_WEIGHT = "/kaggle/input/ubcpytorchwith-classweights-training-fold1of5/Acc0.66_Loss1.0244_epoch16.bin"

In [ ]:
def get_test_file_path(image_id):
    if os.path.exists(f"{TEST_DIR}/{image_id}_thumbnail.png"):
        return f"{TEST_DIR}/{image_id}_thumbnail.png"
    else:
        return f"{ALT_TEST_DIR}/{image_id}.png"

In [ ]:
df = pd.read_csv(f"{ROOT_DIR}/test.csv")
df['file_path'] = df['image_id'].apply(get_test_file_path)
df['label'] = 0 # dummy
df


In [ ]:
df_sub = pd.read_csv(f"{ROOT_DIR}/sample_submission.csv")
df_sub

In [ ]:
encoder = joblib.load( LABEL_ENCODER )

# **Training Configuration**  ⚙️

A training configuration is a vital component of any machine learning or deep learning project, serving as the blueprint that outlines the parameters, settings, and conditions under which a model is trained. This essential document encapsulates the entire training process, providing clarity and reproducibility to the development and deployment of AI systems. Here, we delve into the key elements and considerations that constitute a comprehensive training configuration:

**Model Architecture:** The configuration specifies the architecture of the neural network or machine learning model being trained. It outlines the layers, nodes, and connections that define the model's structure. This includes details like the type of layers (e.g., convolutional, recurrent), activation functions, and any custom layers or modifications.

**Data Preparation:** It outlines the methods and procedures for data preprocessing, augmentation, and normalization. This may include data scaling, one-hot encoding, or image augmentation techniques. Proper data preparation is critical for model convergence and performance.

**Hyperparameters:** The training configuration specifies hyperparameters, which are settings that control the learning process. This includes parameters like learning rate, batch size, epochs, and optimization algorithms (e.g., Adam, SGD). Tinkering with these hyperparameters can significantly impact model training outcomes.

**Loss Function:** The choice of the loss function is pivotal to training. This component of the configuration details the specific loss metric that the model optimizes during training, aligning it with the objectives of the project (e.g., mean squared error for regression, cross-entropy for classification).

**Metrics for Evaluation:** The configuration lists the evaluation metrics used to assess the model's performance during and after training. Common metrics include accuracy, F1-score, mean absolute error (MAE), and mean squared error (MSE).

**Regularization Techniques:** If applicable, regularization techniques such as dropout, L1, or L2 regularization are specified in the configuration to prevent overfitting.

**Checkpointing:** Configuration may include settings for model checkpointing, which periodically saves the model's weights and progress during training. This is essential for resuming training or selecting the best model for deployment.

**Early Stopping:** Parameters for early stopping, based on validation metrics, are often included. This helps prevent overtraining by halting training when the model's performance on validation data plateaus or deteriorates.

**Hardware and Environment:** It mentions the hardware resources utilized during training, including CPU, GPU, or TPUs, as well as the software environment, such as the version of deep learning frameworks (e.g., TensorFlow, PyTorch) and the operating system.

**Batch Processing:** Configuration can also include information on distributed training, specifying whether training is done in a single batch or in mini-batches, and whether it spans multiple GPUs or nodes.

**Data Splits:** The division of data into training, validation, and test sets is outlined in the configuration to ensure proper model assessment and generalization.

**Documentation:** A well-documented training configuration is crucial for reproducibility. It should contain comments and explanations for each parameter and decision made, enabling easy sharing and replication of the training process.

In summary, a training configuration is the comprehensive roadmap that guides the development of machine learning and deep learning models. It plays a pivotal role in achieving reproducibility, scalability, and the successful deployment of AI systems, ensuring that the model can be trained consistently and effectively for various applications.

In [ ]:
class UBCDataset(Dataset):
    def __init__(self, df, transforms=None):
        self.df = df
        self.file_names = df['file_path'].values
        self.labels = df['label'].values
        self.transforms = transforms
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, index):
        img_path = self.file_names[index]
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        label = self.labels[index]
        
        if self.transforms:
            img = self.transforms(image=img)["image"]
            
        return {
            'image': img,
            'label': torch.tensor(label, dtype=torch.long)
        }

# **Augmentations**

In [ ]:
data_transforms = {
    "valid": A.Compose([
        A.Resize(CONFIG['img_size'], CONFIG['img_size']),
        A.Normalize(
                mean=[0.485, 0.456, 0.406], 
                std=[0.229, 0.224, 0.225], 
                max_pixel_value=255.0, 
                p=1.0
            ),
        ToTensorV2()], p=1.)
}

# **GeM Pooling**

In [ ]:
class GeM(nn.Module):
    def __init__(self, p=3, eps=1e-6):
        super(GeM, self).__init__()
        self.p = nn.Parameter(torch.ones(1)*p)
        self.eps = eps

    def forward(self, x):
        return self.gem(x, p=self.p, eps=self.eps)
        
    def gem(self, x, p=3, eps=1e-6):
        return F.avg_pool2d(x.clamp(min=eps).pow(p), (x.size(-2), x.size(-1))).pow(1./p)
        
    def __repr__(self):
        return self.__class__.__name__ + \
                '(' + 'p=' + '{:.4f}'.format(self.p.data.tolist()[0]) + \
                ', ' + 'eps=' + str(self.eps) + ')'

# **Create Model**

<!DOCTYPE html>
<html>
<head>
</head>
<body>
    <h1>EfficientNet</h1>
    <p>
        EfficientNet is a family of convolutional neural network (CNN) architectures designed for efficient and effective deep learning in computer vision tasks. It was introduced in 2019 by researchers at Google AI. EfficientNet models are known for their exceptional performance in image classification tasks while being computationally efficient, making them suitable for a wide range of applications.
    </p>
    <p>
        Key features of EfficientNet include:
    </p>
    <ul>
        <li>Compound Scaling: EfficientNet employs a novel compound scaling method that balances the network's depth, width, and resolution to achieve optimal performance without a significant increase in computational cost.</li>
        <li>Efficient Building Blocks: The architecture incorporates efficient building blocks like depthwise separable convolutions and squeeze-and-excitation blocks to reduce the number of parameters and computational overhead.</li>
        <li>Variants: EfficientNet comes in various variants (e.g., EfficientNet-B0, B1, B2, ..., B7) that offer different trade-offs between model size and accuracy, allowing users to choose the one that suits their specific requirements.</li>
        <li>State-of-the-Art Performance: EfficientNet models have achieved top performance in benchmark datasets such as ImageNet, outperforming many previous CNN architectures with smaller model sizes.</li>
    </ul>
    <p>
        EfficientNet has become a popular choice in the field of computer vision due to its ability to achieve impressive results with fewer parameters, making it practical for deployment on resource-constrained devices and applications.
    </p>
    <p>
        If you are working on image classification or related tasks, considering EfficientNet as part of your deep learning architecture can lead to efficient and accurate results.
    </p>
</body>
</html>


In [ ]:
class UBCModel(nn.Module):
    def __init__(self, model_name, num_classes, pretrained=False, checkpoint_path=None):
        super(UBCModel, self).__init__()
        self.model = timm.create_model(model_name, pretrained=pretrained)

        in_features = self.model.classifier.in_features
        self.model.classifier = nn.Identity()
        self.model.global_pool = nn.Identity()
        self.pooling = GeM()
        self.linear = nn.Linear(in_features, num_classes)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, images):
        features = self.model(images)
        pooled_features = self.pooling(features).flatten(1)
        output = self.linear(pooled_features)
        return output

    
model = UBCModel(CONFIG['model_name'], CONFIG['num_classes'])
model.load_state_dict(torch.load( BEST_WEIGHT ))
model.to(CONFIG['device']);

In [ ]:
model

In [ ]:
test_dataset = UBCDataset(df, transforms=data_transforms["valid"])
test_loader = DataLoader(test_dataset, batch_size=CONFIG['valid_batch_size'], 
                          num_workers=2, shuffle=False, pin_memory=True)

In [ ]:
preds = []
with torch.no_grad():
    bar = tqdm(enumerate(test_loader), total=len(test_loader))
    for step, data in bar:        
        images = data['image'].to(CONFIG["device"], dtype=torch.float)        
        batch_size = images.size(0)
        outputs = model(images)
        _, predicted = torch.max(model.softmax(outputs), 1)
        preds.append( predicted.detach().cpu().numpy() )
preds = np.concatenate(preds).flatten()
pred_labels = encoder.inverse_transform( preds )

In [ ]:
df_sub["label"] = pred_labels
df_sub.to_csv("submission.csv", index=False)

In [ ]:
df_sub


**Dears,**

**I hope this message finds you well. I am excited to share that I am participating in the UBC Ovarian Cancer Subtype Classification and Outlier Detection (UBC-OCEAN) competition, and I need your support!**

**As part of this competition, I have conducted an in-depth Exploratory Data Analysis (EDA) to gain crucial insights into the dataset, which is a fundamental step in developing effective solutions for cancer subtype classification and outlier detection. Now, I am reaching out to request your vote and support for my EDA submission.**

**Your vote can make a significant difference in this competition and help me advance to the next stages. Here's how you can support me:**



**1. Cast Your Vote:**

Visit the competition platform and find my EDA submission.
Click on the "Vote" or "Support" button to cast your vote.

**2. Share with Your Network:**

Spread the word among your friends, family, and colleagues who may be interested in supporting my work.

**3. Provide Feedback:**

If you have any feedback or suggestions on my EDA, please feel free to share them with me. Your input is valuable and can help me improve.
I am committed to making a positive impact in the field of cancer research, and your support will bring me one step closer to achieving that goal.

Thank you for taking the time to read this message, and I genuinely appreciate your support in this competition. Together, we can contribute to the fight against ovarian cancer and advance the field of data-driven healthcare.

If you have any questions or need more information about my EDA, please don't hesitate to reach out to me. Your support means the world to me!

Warm regards,
Jeferson S. Pazze